[Cell 0] 동적 라이브러리 설치

In [ ]:
!pip install -q torch numpy

[Cell 1] 공통 인프라 설정 및 다운로드 함수

In [ ]:
import os
import requests
import mlflow
import json
import boto3
from ultralytics import YOLO, settings
import mlflow.pytorch

settings.update({"mlflow": False})

# 환경 설정
BACKEND_URL = "http://backend:8000"
MLFLOW_TRACKING_URI = "http://mlflow:5000"
S3_ENDPOINT_URL = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://minio:9000")
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# 데이터셋 다운로드 함수
def download_dataset(bucket_name, prefix, local_dir="./data"):
    s3 = boto3.client('s3',
        endpoint_url=S3_ENDPOINT_URL,
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY
    )
    if not os.path.exists(local_dir):
        os.makedirs(local_dir)
    print(f"Downloading: {bucket_name}/{prefix} -> {local_dir}")
    paginator = s3.get_paginator('list_objects_v2')
    for result in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        if 'Contents' in result:
            for obj in result['Contents']:
                key = obj['Key']
                if key.endswith('/'): continue
                local_file_path = os.path.join(local_dir, os.path.relpath(key, prefix))
                os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
                s3.download_file(bucket_name, key, local_file_path)
    return local_dir

[Cell 2] 작업 등록

In [ ]:
PROJECT_ID = 1  
DATASET_PATH = "datasets/1D-CNN/TEP/v1"  
MODEL_ARCHITECTURE = "Autoencoder"

payload = {
    "project_id": PROJECT_ID,
    "model_variant": MODEL_ARCHITECTURE,
    "dataset": DATASET_PATH,
    "params": {"source": "jupyter_notebook"}
}
response = requests.post(f"{BACKEND_URL}/api/v1/jobs/jupyter", json=payload)
response.raise_for_status()
job_info = response.json()

JOB_ID = job_info["id"]
RUN_ID = job_info["run_id"]

print(f"[System] Job Registered! ID: {JOB_ID}, Run ID: {RUN_ID}")

[Cell 3] Autoencoder 학습 로직

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Autoencoder 모델 정의
class TEPAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 32), nn.ReLU(), nn.Linear(32, 16))
        self.decoder = nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, input_dim))

    def forward(self, x):
        return self.decoder(self.encoder(x))

with mlflow.start_run(run_id=RUN_ID):
    print("[System] Autoencoder Training Started...")
    
    try:
        # 1. 데이터셋 다운로드
        bucket, prefix = DATASET_PATH.split('/', 1)
        local_data_path = download_dataset(bucket, prefix, local_dir=f"./data/job_{JOB_ID}")
        print(f"[System] Dataset downloaded at: {local_data_path}")

        # 2. X.npy, y.npy 로드 및 차원 맞추기
        X = np.load(os.path.join(local_data_path, "X.npy"))
        y = np.load(os.path.join(local_data_path, "y.npy"))

        if len(X.shape) == 3:
            samples, timesteps, features = X.shape
            X = X.reshape(samples, timesteps * features)

        # 3. 비지도 학습을 위해 '정상' 데이터만 추출 (보통 클래스 0을 정상으로 간주)
        y_flat = y.flatten()
        X_normal = X[y_flat == 0]
        print(f"[System] Filtered normal data shape for Autoencoder: {X_normal.shape}")

        # PyTorch 텐서 변환 및 데이터로더 생성
        X_tensor = torch.tensor(X_normal, dtype=torch.float32)
        dataset = TensorDataset(X_tensor)
        dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

        # 4. 모델 세팅
        input_dim = X.shape[1]
        model = TEPAutoencoder(input_dim)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        epochs = 10
        mlflow.log_params({"epochs": epochs, "learning_rate": 0.001, "batch_size": 64})

        # 5. 학습 루프 진행
        print("[System] Model fitting...")
        for epoch in range(epochs):
            epoch_loss = 0.0
            for batch in dataloader:
                inputs = batch[0]
                outputs = model(inputs)
                loss = criterion(outputs, inputs)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

            avg_loss = epoch_loss / len(dataloader)
            # MLflow에 에폭마다 Loss 기록 (그래프 생성용)
            mlflow.log_metric("train_loss", avg_loss, step=epoch+1)
            
            if (epoch + 1) % 2 == 0:
                print(f"[System] Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

        # 6. 완성된 모델 MLflow에 저장
        mlflow.pytorch.log_model(
            pytorch_model=model, 
            artifact_path="model", 
            registered_model_name=f"Jupyter_AE_Job_{JOB_ID}"
        )

        print("[System] Training Complete!")
        status_to_report = "FINISHED"
        message = "Autoencoder training completed successfully."

    except Exception as e:
        print(f"[System] Training Failed: {e}")
        status_to_report = "FAILED"
        message = str(e)
        mlflow.end_run(status='FAILED')

[Cell 4] Webhook 전송

In [ ]:
webhook_url = f"{BACKEND_URL}/api/v1/jobs/{JOB_ID}/complete"
resp = requests.post(webhook_url, json={"status": status_to_report, "message": message})

if resp.status_code == 200:
    print(f"[System] Webhook sent successfully. Job {JOB_ID} is now {status_to_report}.")
else:
    print(f"[Error] Webhook failed: {resp.status_code}")